# Solid-State Battery Design Comparison

Compare 4 solid-state cell designs across battery, ceramic, polymer, and metal domains.
Includes synthesis planning for solid electrolytes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd

json_path = '../showcase/outputs/solid_state_designs.json'
csv_path = '../showcase/outputs/solid_state_synthesis.csv'

if not os.path.exists(json_path):
    print('Generating data...')
    from showcase.solid_state_design import main
    main()

with open(json_path) as f:
    data = json.load(f)

synth_df = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()

print(f"Designs compared: {len(data['designs'])}")
for d in data['ranking']:
    print(f"  {d['name']}: {d['score']:.4f} {'VIABLE' if d['viable'] else 'NOT VIABLE'}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Side-by-side bar chart of designs
designs = data['designs']
names = [d['name'].split(':')[1].strip() if ':' in d['name'] else d['name'] for d in designs]
scores = [d['overall_score'] for d in designs]
colors = ['green' if d['viable'] else 'red' for d in designs]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, scores, color=colors, edgecolor='black', alpha=0.8)
ax.axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Viability threshold')
ax.set_xlabel('Overall Score')
ax.set_title('Solid-State Battery Design Comparison')
ax.legend()

for bar, score in zip(bars, scores):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Interface breakdown for each design
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, design in zip(axes.flat, designs):
    if 'interfaces' not in design:
        continue
    ifaces = design['interfaces']
    labels = list(ifaces.keys())
    vals = [ifaces[k]['score'] for k in labels]
    short_labels = [l.replace('<->', '\n') for l in labels]
    colors_i = ['green' if ifaces[k]['compatible'] else 'red' for k in labels]

    ax.bar(range(len(labels)), vals, color=colors_i, edgecolor='black', alpha=0.7)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(short_labels, fontsize=8)
    ax.axhline(0.5, color='orange', linestyle='--')
    ax.set_ylim(0, 1)
    ax.set_title(design['name'].split(':')[1].strip() if ':' in design['name'] else design['name'],
                 fontsize=10)
    ax.set_ylabel('Score')

plt.suptitle('Interface Scores per Design (orange = viability threshold)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Synthesis comparison for solid electrolytes
if not synth_df.empty:
    print('Solid Electrolyte Synthesis Comparison:')
    print(synth_df[['target', 'route_name', 'composite', 'precursor_cost_usd',
                    'total_time_hours', 'safety_score']].to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, col, label in zip(axes,
                               ['precursor_cost_usd', 'total_time_hours', 'safety_score'],
                               ['Cost ($/g)', 'Time (hours)', 'Safety Score']):
        ax.bar(synth_df['target'], synth_df[col], edgecolor='black', alpha=0.7)
        ax.set_title(label)
        ax.set_ylabel(label)
        for i, v in enumerate(synth_df[col]):
            ax.text(i, v + 0.02 * max(synth_df[col]), f'{v:.2f}',
                    ha='center', fontsize=9)

    plt.suptitle('Solid Electrolyte Synthesis: Cost vs Time vs Safety')
    plt.tight_layout()
    plt.show()
else:
    print('No synthesis data available.')